In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import (
    HeteroConv,
    GCNConv,
    GATConv,
    global_mean_pool,
    global_max_pool,
)
from torch_geometric.data import HeteroData, Batch
from typing import Dict, List, Tuple, Optional
import math
import sys

sys.path.append("..")  # To ensure parent directory is in path


class NodeEmbedding(nn.Module):
    """Separate embedding layers for pixel and MPPC nodes."""

    def __init__(self, input_dims: Dict[str, int], hidden_dim: int = 128):
        super().__init__()
        self.embeddings = nn.ModuleDict()

        for node_type, input_dim in input_dims.items():
            self.embeddings[node_type] = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dim, hidden_dim),
            )

    def forward(self, x_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        return {
            node_type: self.embeddings[node_type](x) for node_type, x in x_dict.items()
        }


class HeteroGNNLayer(nn.Module):
    """Single heterogeneous GNN layer with attention."""

    def __init__(
        self,
        hidden_dim: int,
        edge_types: List[Tuple[str, str, str]],
        use_attention: bool = True,
        heads: int = 4,
    ):
        super().__init__()

        convs = {}
        for edge_type in edge_types:
            if use_attention:
                convs[edge_type] = GATConv(
                    hidden_dim,
                    hidden_dim // heads,
                    heads=heads,
                    dropout=0.1,
                    add_self_loops=False,
                )
            else:
                convs[edge_type] = GCNConv(hidden_dim, hidden_dim, add_self_loops=False)

        self.conv = HeteroConv(convs, aggr="mean")
        self.norm = nn.ModuleDict(
            {node_type: nn.LayerNorm(hidden_dim) for node_type in ["pixel", "mppc"]}
        )

    def forward(
        self,
        x_dict: Dict[str, torch.Tensor],
        edge_index_dict: Dict[Tuple[str, str, str], torch.Tensor],
    ) -> Dict[str, torch.Tensor]:

        x_dict = self.conv(x_dict, edge_index_dict)

        # Apply normalization and residual connection
        for node_type in x_dict:
            x_dict[node_type] = self.norm[node_type](x_dict[node_type])

        return x_dict


class EdgeClassifier(nn.Module):
    """Edge classification head for track prediction."""

    def __init__(self, hidden_dim: int, edge_types: List[Tuple[str, str, str]]):
        super().__init__()

        self.edge_classifiers = nn.ModuleDict()
        for edge_type in edge_types:
            edge_key = f"{edge_type[0]}_{edge_type[2]}"
            self.edge_classifiers[edge_key] = nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dim, 1),
                nn.Sigmoid(),
            )

    def forward(
        self,
        x_dict: Dict[str, torch.Tensor],
        edge_index_dict: Dict[Tuple[str, str, str], torch.Tensor],
    ) -> Dict[str, torch.Tensor]:

        edge_predictions = {}

        for edge_type, edge_index in edge_index_dict.items():
            src_type, _, dst_type = edge_type
            edge_key = f"{src_type}_{dst_type}"

            if edge_key in self.edge_classifiers:
                # Get source and destination node embeddings
                src_emb = x_dict[src_type][edge_index[0]]
                dst_emb = x_dict[dst_type][edge_index[1]]

                # Concatenate and classify
                edge_features = torch.cat([src_emb, dst_emb], dim=1)
                edge_predictions[edge_key] = self.edge_classifiers[edge_key](
                    edge_features
                )

        return edge_predictions


class TimeSliceEncoder(nn.Module):
    """Encodes individual 8ns time slices into embeddings."""

    def __init__(
        self,
        hidden_dim: int,
        num_gnn_layers: int = 3,
        edge_types: List[Tuple[str, str, str]] = None,
    ):
        super().__init__()

        # Node embedding
        self.node_embedding = NodeEmbedding(
            input_dims={
                "pixel": 4,
                "mppc": 5,
            },  # pixel: x,y,z,layer; mppc: x,y,z,layer,time
            hidden_dim=hidden_dim,
        )

        # Heterogeneous GNN layers
        self.gnn_layers = nn.ModuleList(
            [
                HeteroGNNLayer(hidden_dim, edge_types, use_attention=(i > 0))
                for i in range(num_gnn_layers)
            ]
        )

        # Graph-level pooling
        self.global_pool = nn.ModuleDict(
            {
                "pixel": nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU()
                ),
                "mppc": nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU()
                ),
            }
        )

        # Edge classifier
        self.edge_classifier = (
            EdgeClassifier(hidden_dim, edge_types) if edge_types else None
        )

    def forward(
        self, batch: HeteroData, return_edge_pred: bool = False
    ) -> Tuple[torch.Tensor, Optional[Dict]]:
        # Node embedding
        x_dict = self.node_embedding(batch.x_dict)

        # Store initial embeddings for residual connections
        initial_x_dict = {k: v.clone() for k, v in x_dict.items()}

        # Apply GNN layers
        for i, gnn_layer in enumerate(self.gnn_layers):
            new_x_dict = gnn_layer(x_dict, batch.edge_index_dict)

            # Residual connection every 2 layers
            if i > 0 and i % 2 == 0:
                for node_type in new_x_dict:
                    new_x_dict[node_type] = new_x_dict[node_type] + x_dict[node_type]

            x_dict = new_x_dict

        # Global pooling for each node type
        graph_embeddings = []

        for node_type in ["pixel", "mppc"]:
            if node_type in x_dict and x_dict[node_type].size(0) > 0:
                # Use both mean and max pooling
                node_emb = x_dict[node_type]
                pooled = self.global_pool[node_type](node_emb)

                # Global pooling (handle batch dimension)
                if hasattr(batch, f"{node_type}_batch"):
                    batch_idx = getattr(batch, f"{node_type}_batch")
                    mean_pool = global_mean_pool(pooled, batch_idx)
                    max_pool = global_max_pool(pooled, batch_idx)
                else:
                    # Single graph case
                    mean_pool = pooled.mean(dim=0, keepdim=True)
                    max_pool = pooled.max(dim=0, keepdim=True)[0]

                graph_embeddings.append(torch.cat([mean_pool, max_pool], dim=1))
            else:
                # Handle missing node types
                hidden_dim = self.global_pool["pixel"][0].in_features
                device = next(self.parameters()).device
                empty_emb = torch.zeros(1, hidden_dim, device=device)
                graph_embeddings.append(torch.cat([empty_emb, empty_emb], dim=1))

        # Combine pixel and MPPC embeddings
        combined_embedding = torch.cat(graph_embeddings, dim=1)

        # Edge predictions if requested
        edge_predictions = None
        if return_edge_pred and self.edge_classifier:
            edge_predictions = self.edge_classifier(x_dict, batch.edge_index_dict)

        return combined_embedding, edge_predictions


class PositionalEncoding(nn.Module):
    """Positional encoding for time sequence."""

    def __init__(self, d_model: int, max_len: int = 8):  # Max 8 time slices in 64ns
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch_size, seq_len, d_model]
        seq_len = x.size(1)
        return x + self.pe[:seq_len, :].unsqueeze(0)


class TemporalAggregator(nn.Module):
    """Aggregates time slice embeddings into event-level representation."""

    def __init__(self, hidden_dim: int, max_time_slices: int = 8):
        super().__init__()

        self.pos_encoding = PositionalEncoding(hidden_dim, max_time_slices)

        # Transformer for temporal modeling
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=8,
            dim_feedforward=hidden_dim * 2,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # Attention-based final aggregation
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, batch_first=True
        )

        self.final_norm = nn.LayerNorm(hidden_dim)

    def forward(
        self, time_slice_embeddings: torch.Tensor, mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        # time_slice_embeddings: [batch_size, num_time_slices, hidden_dim]

        # Add positional encoding
        x = self.pos_encoding(time_slice_embeddings)

        # Create attention mask for variable length sequences
        if mask is not None:
            # mask: [batch_size, num_time_slices], True for valid positions
            attn_mask = ~mask  # Invert for transformer (True = ignore)
        else:
            attn_mask = None

        # Apply transformer
        x = self.transformer(x, src_key_padding_mask=attn_mask)

        # Final attention aggregation
        if mask is not None:
            # Use valid time slices for attention
            valid_lengths = mask.sum(dim=1)
            event_embeddings = []

            for i, length in enumerate(valid_lengths):
                if length > 0:
                    query = x[i : i + 1, :length, :]  # [1, valid_length, hidden_dim]
                    attn_out, _ = self.attention(query, query, query)
                    event_emb = attn_out.mean(dim=1)  # [1, hidden_dim]
                else:
                    # Handle empty sequences
                    device = x.device
                    event_emb = torch.zeros(1, x.size(-1), device=device)
                event_embeddings.append(event_emb)

            event_representation = torch.cat(event_embeddings, dim=0)
        else:
            # Simple case: use mean pooling
            attn_out, _ = self.attention(x, x, x)
            event_representation = attn_out.mean(dim=1)

        return self.final_norm(event_representation)


class Mu3EEventClassifier(nn.Module):
    """Complete Mu3E event classification model."""

    def __init__(
        self,
        hidden_dim: int = 256,
        num_gnn_layers: int = 4,
        num_classes: int = 2,  # signal vs background
        max_time_slices: int = 8,
        use_edge_classification: bool = True,
    ):
        super().__init__()

        # Define edge types for heterogeneous graph
        self.edge_types = [
            ("pixel", "to", "pixel"),
            ("mppc", "to", "mppc"),
            ("pixel", "to", "mppc"),
            ("mppc", "to", "pixel"),
        ]

        # Time slice encoder
        self.time_slice_encoder = TimeSliceEncoder(
            hidden_dim=hidden_dim,
            num_gnn_layers=num_gnn_layers,
            edge_types=self.edge_types if use_edge_classification else None,
        )

        # Temporal aggregator
        self.temporal_aggregator = TemporalAggregator(
            hidden_dim=hidden_dim, max_time_slices=max_time_slices
        )

        # Event classifier
        self.event_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 4, num_classes),
        )

        self.use_edge_classification = use_edge_classification

    def forward(
        self, batch_list: List[HeteroData], return_edge_predictions: bool = None
    ) -> Dict[str, torch.Tensor]:
        """
        Args:
            batch_list: List of HeteroData batches, one per time slice
            return_edge_predictions: Whether to return edge predictions

        Returns:
            Dictionary with 'event_logits' and optionally 'edge_predictions'
        """

        if return_edge_predictions is None:
            return_edge_predictions = self.use_edge_classification

        batch_size = len(batch_list)
        time_slice_embeddings = []
        all_edge_predictions = []

        # Process each time slice
        for batch in batch_list:
            if batch is not None:
                embedding, edge_pred = self.time_slice_encoder(
                    batch, return_edge_pred=return_edge_predictions
                )
                time_slice_embeddings.append(embedding)
                if edge_pred:
                    all_edge_predictions.append(edge_pred)
            else:
                # Handle empty time slices
                device = next(self.parameters()).device
                empty_emb = torch.zeros(
                    batch_size,
                    self.time_slice_encoder.global_pool["pixel"][0].out_features * 4,
                    device=device,
                )
                time_slice_embeddings.append(empty_emb)

        # Stack time slice embeddings
        if time_slice_embeddings:
            stacked_embeddings = torch.stack(
                time_slice_embeddings, dim=1
            )  # [batch, time, hidden]

            # Create mask for valid time slices
            mask = torch.ones(
                stacked_embeddings.size()[:2],
                dtype=torch.bool,
                device=stacked_embeddings.device,
            )

            # Aggregate temporal information
            event_embedding = self.temporal_aggregator(stacked_embeddings, mask)

            # Event classification
            event_logits = self.event_classifier(event_embedding)
        else:
            # Handle case with no valid time slices
            device = next(self.parameters()).device
            event_logits = torch.zeros(batch_size, 2, device=device)

        results = {"event_logits": event_logits}

        if return_edge_predictions and all_edge_predictions:
            # Combine edge predictions from all time slices
            combined_edge_pred = {}
            for edge_pred in all_edge_predictions:
                for key, pred in edge_pred.items():
                    if key not in combined_edge_pred:
                        combined_edge_pred[key] = []
                    combined_edge_pred[key].append(pred)

            # Concatenate predictions from different time slices
            for key in combined_edge_pred:
                combined_edge_pred[key] = torch.cat(combined_edge_pred[key], dim=0)

            results["edge_predictions"] = combined_edge_pred

        return results


def create_mu3e_model(config: Dict = None) -> Mu3EEventClassifier:
    """Factory function to create Mu3E model with default or custom config."""

    default_config = {
        "hidden_dim": 32,
        "num_gnn_layers": 4,
        "num_classes": 2,
        "max_time_slices": 8,
        "use_edge_classification": True,
    }


    return Mu3EEventClassifier(**default_config)


# Example usage and training utilities
class Mu3ELoss(nn.Module):
    """Multi-task loss for event and edge classification."""

    def __init__(self, event_weight: float = 1.0, edge_weight: float = 0.5):
        super().__init__()
        self.event_weight = event_weight
        self.edge_weight = edge_weight
        self.event_loss = nn.CrossEntropyLoss()
        self.edge_loss = nn.BCELoss()

    def forward(self, predictions: Dict, targets: Dict) -> Dict[str, torch.Tensor]:
        losses = {}
        total_loss = 0

        # Event classification loss
        if "event_logits" in predictions and "event_labels" in targets:
            event_loss = self.event_loss(
                predictions["event_logits"], targets["event_labels"]
            )
            losses["event_loss"] = event_loss
            total_loss += self.event_weight * event_loss

        # Edge classification loss
        if "edge_predictions" in predictions and "edge_labels" in targets:
            edge_loss = 0
            num_edge_types = 0

            for edge_type, pred in predictions["edge_predictions"].items():
                if edge_type in targets["edge_labels"]:
                    edge_loss += self.edge_loss(
                        pred.squeeze(), targets["edge_labels"][edge_type].float()
                    )
                    num_edge_types += 1

            if num_edge_types > 0:
                edge_loss /= num_edge_types
                losses["edge_loss"] = edge_loss
                total_loss += self.edge_weight * edge_loss

        losses["total_loss"] = total_loss
        return losses

In [2]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch_geometric.data import Batch
from typing import List, Dict, Tuple, Optional


# Assuming you have the previous architecture imported
# from mu3e_gnn_architecture import Mu3EEventClassifier, Mu3ELoss, create_mu3e_model


# Integration with your existing graph processing
class Mu3EDataProcessor:
    """Processes Mu3E events into time-sliced graph batches."""

    def __init__(self, processor, max_time_slices: int = 8):
        self.processor = processor  # Your existing EventProcessor
        self.max_time_slices = max_time_slices

    def process_event_to_time_slices(
        self, X_pixel , X_mppc, label: int
    ) -> Tuple[List[Batch], Dict]:
        """
        Process single event into time-sliced batches.

        Returns:
            - List of Batch objects, one per time slice
            - Dictionary with event-level and edge-level targets
        """

        # Convert to your existing graph format
        graphs = self.processor.process_single_event(X_pixel=X_pixel, X_mppc=X_mppc)

        if not graphs:
            return [], {}

        # Group graphs by time slice and create batches
        time_slice_batches = []
        edge_targets = {
            "pixel_pixel": [],
            "pixel_mppc": [],
            "mppc_pixel": [],
            "mppc_mppc": [],
        }

        for graph in graphs:
            # Create batch from single graph
            batch = Batch.from_data_list([graph])
            time_slice_batches.append(batch)

            # Collect edge labels for edge classification
            if hasattr(graph, "edge_labels"):
                # Map your edge types to target dictionary
                for edge_type in graph.edge_types:
                    edge_key = f"{edge_type[0]}_{edge_type[2]}"
                    if edge_key in edge_targets and hasattr(
                        graph[edge_type], "edge_labels"
                    ):
                        edge_targets[edge_key].append(graph[edge_type].edge_labels)

        # Pad to max_time_slices if needed
        while len(time_slice_batches) < self.max_time_slices:
            time_slice_batches.append(None)  # Model handles None gracefully

        # Concatenate edge labels
        for key in edge_targets:
            if edge_targets[key]:
                edge_targets[key] = torch.cat(edge_targets[key], dim=0)
            else:
                edge_targets[key] = torch.tensor([])  # Empty tensor

        targets = {
            "event_labels": torch.tensor(label, dtype=torch.long),
            "edge_labels": edge_targets,
        }

        return time_slice_batches[: self.max_time_slices], targets

from src.torch.pre_processing.graph_batching import HeteroGraphBuilder, EventProcessor

class Mu3EDataset(torch.utils.data.Dataset):
    """Dataset for Mu3E events."""

    def __init__(self, X_pixel, X_mppc, labels):
        super().__init__()
    
        processor = EventProcessor(HeteroGraphBuilder())
        
        self.X_pixel = X_pixel if isinstance(X_pixel, torch.Tensor) else torch.tensor(X_pixel, dtype=torch.float32)
        self.X_mppc = X_mppc if isinstance(X_mppc, torch.Tensor) else torch.tensor(X_mppc, dtype=torch.float32)
        self.labels = labels if isinstance(labels, torch.Tensor) else torch.tensor(labels, dtype=torch.float32)
        self.data_processor = Mu3EDataProcessor(processor)

        # Filter viable events (from your existing code)
        from src.torch.pre_processing.graph_batching import EventFilter

        self.viable_indices = processor.get_viable_event_indices(
            self.X_pixel, self.X_mppc, min_tracks=2
        )

        print(f"Dataset: {len(self.viable_indices)} viable events out of {len(labels)}")

    def __len__(self):
        return len(self.viable_indices)

    def __getitem__(self, idx):
        actual_idx = self.viable_indices[idx]

        time_slices, targets = self.data_processor.process_event_to_time_slices(
            self.X_pixel[actual_idx], self.X_mppc[actual_idx], self.labels[actual_idx]
        )

        return time_slices, targets

class GraphLoader:
    """Custom DataLoader for Mu3E events."""

    def __init__(self, dataset: Mu3EDataset, batch_size: int = 4, shuffle: bool = True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        indices = list(range(len(self.dataset)))
        if self.shuffle:
            np.random.shuffle(indices)
            
        for start_idx in range(0, len(self.dataset), self.batch_size):
            batch_indices = indices[start_idx : start_idx + self.batch_size]
            batch_time_slices = []
            batch_targets = []
            for idx in batch_indices:
                time_slices, targets = self.dataset[idx]
                batch_time_slices.append(time_slices)
                batch_targets.append(targets)

            # Transpose to get list of time slices across the batch
            transposed_time_slices = list(
                zip(*batch_time_slices)
            )  # List of time slices, each containing a list of graphs

            yield transposed_time_slices, batch_targets

    def __len__(self):
        return math.ceil(len(self.dataset) / self.batch_size)

class Mu3ETrainer:
    """Training pipeline for Mu3E event classification."""

    def __init__(self, model: Mu3EEventClassifier, config: Dict):
        self.model = model
        self.config = config

        # Loss and optimizer
        self.criterion = Mu3ELoss(
            event_weight=config.pop("event_weight", 1.0),
            edge_weight=config.pop("edge_weight", 0.3),
        )

        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.pop("learning_rate", 1e-3),
            weight_decay=config.pop("weight_decay", 1e-4),
        )

        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.5, patience=5
        )

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def train_epoch(self, train_loader: DataLoader) -> Dict[str, float]:
        """Train one epoch."""
        self.model.train()

        total_loss = 0
        event_correct = 0
        total_events = 0

        for batch_idx, (time_slices, targets) in enumerate(train_loader):
            self.optimizer.zero_grad()

            # Move data to device
            time_slices = [
                ts.to(self.device) if ts is not None else None for ts in time_slices
            ]

            targets_device = {
                "event_labels": targets["event_labels"].to(self.device),
                "edge_labels": {
                    k: v.to(self.device) for k, v in targets["edge_labels"].items()
                },
            }

            # Forward pass
            predictions = self.model(time_slices, return_edge_predictions=True)

            # Compute loss
            losses = self.criterion(predictions, targets_device)
            loss = losses["total_loss"]

            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            # Statistics
            total_loss += loss.item()

            # Event accuracy
            event_preds = torch.argmax(predictions["event_logits"], dim=1)
            event_correct += (
                (event_preds == targets_device["event_labels"]).sum().item()
            )
            total_events += targets_device["event_labels"].size(0)

        return {
            "train_loss": total_loss / len(train_loader),
            "train_accuracy": event_correct / total_events,
        }

    def validate(self, val_loader: DataLoader) -> Dict[str, float]:
        """Validate model."""
        self.model.eval()

        total_loss = 0
        event_correct = 0
        total_events = 0

        with torch.no_grad():
            for time_slices, targets in val_loader:
                # Move to device
                time_slices = [
                    ts.to(self.device) if ts is not None else None for ts in time_slices
                ]

                targets_device = {
                    "event_labels": targets["event_labels"].to(self.device),
                    "edge_labels": {
                        k: v.to(self.device) for k, v in targets["edge_labels"].items()
                    },
                }

                # Forward pass
                predictions = self.model(time_slices, return_edge_predictions=True)

                # Compute loss
                losses = self.criterion(predictions, targets_device)
                total_loss += losses["total_loss"].item()

                # Event accuracy
                event_preds = torch.argmax(predictions["event_logits"], dim=1)
                event_correct += (
                    (event_preds == targets_device["event_labels"]).sum().item()
                )
                total_events += targets_device["event_labels"].size(0)

        return {
            "val_loss": total_loss / len(val_loader),
            "val_accuracy": event_correct / total_events,
        }

    def train(
        self, train_loader: DataLoader, val_loader: DataLoader, num_epochs: int
    ) -> Dict[str, List[float]]:
        """Full training loop."""

        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
        best_val_loss = float("inf")

        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs}")

            # Training
            train_metrics = self.train_epoch(train_loader)

            # Validation
            val_metrics = self.validate(val_loader)

            # Update history
            history["train_loss"].append(train_metrics["train_loss"])
            history["train_acc"].append(train_metrics["train_accuracy"])
            history["val_loss"].append(val_metrics["val_loss"])
            history["val_acc"].append(val_metrics["val_accuracy"])

            # Learning rate scheduling
            self.scheduler.step(val_metrics["val_loss"])

            # Save best model
            if val_metrics["val_loss"] < best_val_loss:
                best_val_loss = val_metrics["val_loss"]
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": self.model.state_dict(),
                        "optimizer_state_dict": self.optimizer.state_dict(),
                        "val_loss": best_val_loss,
                        "config": self.config,
                    },
                    "best_mu3e_model.pt",
                )

            print(
                f"Train Loss: {train_metrics['train_loss']:.4f}, "
                f"Train Acc: {train_metrics['train_accuracy']:.4f}"
            )
            print(
                f"Val Loss: {val_metrics['val_loss']:.4f}, "
                f"Val Acc: {val_metrics['val_accuracy']:.4f}"
            )

        return history

In [3]:
DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_with_layer_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_with_layer_mppc_spacetime.npy"

BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_with_layer_pixel_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_with_layer_mppc_spacetime.npy"


sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)


X_pixel = np.concatenate([sig_pixel_spacetime, bg_pixel_spacetime], axis=0)
X_mppc = np.concatenate([sig_mppc_spacetime, bg_mppc_spacetime], axis=0)

y = np.array([1] * len(sig_pixel_spacetime) + [0] * len(bg_pixel_spacetime))
del sig_pixel_spacetime, sig_mppc_spacetime, bg_pixel_spacetime, bg_mppc_spacetime

from sklearn.model_selection import train_test_split

X_pixel_train, X_pixel_val, X_mppc_train, X_mppc_val, y_train, y_val = train_test_split(
    X_pixel, X_mppc, y, test_size=0.2, random_state=42, stratify=y
)
del X_pixel, X_mppc, y


# Convert to torch tensorsX_pixel_train = torch.tensor(X_pixel_train, dtype=torch.float32)
X_pixel_train = torch.tensor(X_pixel_train, dtype=torch.float32)
X_mppc_train = torch.tensor(X_mppc_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_pixel_val = torch.tensor(X_pixel_val, dtype=torch.float32)
X_mppc_val = torch.tensor(X_mppc_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

from src.torch.pre_processing.graph_batching import HeteroGraphBuilder


In [4]:

train_dataset = Mu3EDataset(
    X_pixel_train, X_mppc_train, y_train
)
val_dataset = Mu3EDataset(
    X_pixel_val, X_mppc_val, y_val
)

train_loader = GraphLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = GraphLoader(val_dataset, batch_size=4, shuffle=False)

Dataset: 288885 viable events out of 293024
Dataset: 72214 viable events out of 73256


In [9]:
for times_slice, data in train_loader:
    print(len(times_slice), len(data))
    break

8 4


/var/folders/_g/_c450dkj4j5djz911r45hnlc0000gn/T/ipykernel_71010/3747541771.py:73: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "event_labels": torch.tensor(label, dtype=torch.long),


In [6]:
"""Example of complete training pipeline."""

# Configuration
config = {
    "hidden_dim": 64,
    "num_gnn_layers": 4,
    "num_classes": 2,
    "max_time_slices": 8,
    "use_edge_classification": True,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "event_weight": 1.0,
    "edge_weight": 0.3,
    "batch_size": 8,  # Small batch size due to memory constraints
    "num_epochs": 20,
}

# Load your data (replace with your actual data loading)
# X_pixel_train, X_mppc_train, y_train = load_training_data()
# X_pixel_val, X_mppc_val, y_val = load_validation_data()

# Create your existing processor
from src.torch.pre_processing.graph_batching import create_hetero_processor

processor = create_hetero_processor(connect_layers=True, mppc_timing_cutoff=0.2)

# Create datasets
# train_dataset = Mu3EDataset(X_pixel_train, X_mppc_train, y_train, processor)
# val_dataset = Mu3EDataset(X_pixel_val, X_mppc_val, y_val, processor)

# Create data loaders
# train_loader = DataLoader(train_dataset, batch_size=config['batch_size'],
#                          shuffle=True, collate_fn=custom_collate_fn)
# val_loader = DataLoader(val_dataset, batch_size=config['batch_size'],
#                        shuffle=False, collate_fn=custom_collate_fn)

# Create model
model = create_mu3e_model(config)
    
print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

# Create trainer
trainer = Mu3ETrainer(model, config)

# Train model
history = trainer.train(train_loader, val_loader, config['num_epochs'])

print("Training pipeline ready!")

Model created with 52,334 parameters

Epoch 1/20


/var/folders/_g/_c450dkj4j5djz911r45hnlc0000gn/T/ipykernel_71010/3747541771.py:73: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "event_labels": torch.tensor(label, dtype=torch.long),


AttributeError: 'tuple' object has no attribute 'to'